In [ ]:
#============================================================
# Experiment 1: Support Vector Machine (SVM) from Scratch
# Synthetic Student Performance Dataset
#============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, classification_report
)

# Step 1: Create Synthetic Dataset
data = {
    "Study_Hours": [1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 2, 3, 4, 6, 7],
    "Attendance": [50, 55, 60, 58, 65, 70, 72, 75, 78, 80, 85, 88, 90, 92, 95, 52, 62, 68, 82, 87],
    "Class": [-1, -1, -1, -1, -1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, 1]
}
df = pd.DataFrame(data)
print("Student Performance Dataset:")
print(df)

# Step 2: Display Dataset Information
print("\nDataset Shape:")
print(df.shape)
print("\nDataset Description:")
print(df.describe())
print("\nClass Distribution:")
print(df["Class"].value_counts())

# Step 3: Separate Features and Target
X = df[["Study_Hours", "Attendance"]].values
y = df["Class"].values

# Step 4: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# Step 5: Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 6: Define SVM Class from Scratch
class SVMFromScratch:

    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.lambda_param = lambda_param
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []

    # Fit Function (Hinge Loss + Gradient Descent)
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)   # Initialize weights
        self.bias = 0                         # Initialize bias

        for iteration in range(self.n_iterations):
            total_loss = 0

            for index, x_i in enumerate(X):
                condition = y[index] * (np.dot(x_i, self.weights) + self.bias) >= 1

                # Case 1: Margin condition satisfied
                if condition:
                    dw = 2 * self.lambda_param * self.weights
                    db = 0
                    hinge_loss = 0

                # Case 2: Margin condition violated
                else:
                    dw = 2 * self.lambda_param * self.weights - y[index] * x_i
                    db = -y[index]
                    hinge_loss = 1 - y[index] * (np.dot(x_i, self.weights) + self.bias)

                # Update weights and bias
                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db
                total_loss += max(0, hinge_loss)

            # Regularization loss + average total loss
            regularization_loss = self.lambda_param * np.dot(self.weights, self.weights)
            average_loss = total_loss / n_samples + regularization_loss
            self.loss_history.append(average_loss)

    # Predict Function
    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        predictions = np.where(linear_output >= 0, 1, -1)
        return predictions

# Step 7: Create SVM Model
svm = SVMFromScratch(learning_rate=0.01, lambda_param=0.01, n_iterations=1000)

# Step 8: Train SVM Model
svm.fit(X_train_scaled, y_train)
print("\nSVM Training Completed Successfully.")

# Step 9: Display Model Parameters
print("\n===================================")
print("SVM MODEL PARAMETERS")
print("===================================")
print("Weights:", svm.weights)
print("Bias:", svm.bias)

# Step 10: Make Predictions
y_pred = svm.predict(X_test_scaled)

# Step 11: Actual vs Predicted
results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
print("\n===================================")
print("ACTUAL VS PREDICTED")
print("===================================")
print(results)

# Steps 12-15: Accuracy, Precision, Recall, F1
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

# Step 16: Display Performance
print("\n===================================")
print("SVM CLASSIFICATION PERFORMANCE")
print("===================================")
print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

# Step 17: Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])
print("\n===================================")
print("CONFUSION MATRIX")
print("===================================")
print(cm)

# Step 18: Classification Report
print("\n===================================")
print("CLASSIFICATION REPORT")
print("===================================")
print(classification_report(
    y_test, y_pred, labels=[-1, 1],
    target_names=["Fail", "Pass"], zero_division=0
))

# Step 19: Plot Training Loss
plt.figure(figsize=(8, 5))
plt.plot(svm.loss_history)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("SVM Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 20: Plot Decision Boundary
X_all_scaled = scaler.transform(X)
plt.figure(figsize=(8, 6))

fail_indices = (y == -1)
plt.scatter(X_all_scaled[fail_indices, 0], X_all_scaled[fail_indices, 1],
            marker="o", label="Fail (-1)")

pass_indices = (y == 1)
plt.scatter(X_all_scaled[pass_indices, 0], X_all_scaled[pass_indices, 1],
            marker="x", label="Pass (+1)")

# Decision Boundary: w1*x1 + w2*x2 + b = 0
x_values = np.linspace(X_all_scaled[:, 0].min() - 1, X_all_scaled[:, 0].max() + 1, 100)
y_values = -((svm.weights[0] * x_values + svm.bias) / svm.weights[1])
plt.plot(x_values, y_values, label="Decision Boundary")

# Margin Lines
margin_positive = (1 - svm.weights[0] * x_values - svm.bias) / svm.weights[1]
margin_negative = (-1 - svm.weights[0] * x_values - svm.bias) / svm.weights[1]
plt.plot(x_values, margin_positive, linestyle="--", label="Positive Margin")
plt.plot(x_values, margin_negative, linestyle="--", label="Negative Margin")

plt.xlabel("Scaled Study Hours")
plt.ylabel("Scaled Attendance")
plt.title("SVM Decision Boundary")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 21: Predict New Student
new_student = np.array([[5, 78]])
new_student_scaled = scaler.transform(new_student)
new_prediction = svm.predict(new_student_scaled)

print("\n===================================")
print("NEW STUDENT PREDICTION")
print("===================================")
print("Study Hours:", new_student[0][0])
print("Attendance:", new_student[0][1])
if new_prediction[0] == 1:
    print("Predicted Class: Pass")
else:
    print("Predicted Class: Fail")

In [ ]:
#============================================================
# Experiment 2: Support Vector Machine Using a Real-World Dataset
# Breast Cancer Dataset - Using Scikit-learn (RBF Kernel)
#============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

# Step 1: Load Dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names
target_names = cancer.target_names

# Step 2: Convert Dataset to DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y

print("Dataset Shape:")
print(df.shape)
print("\nFirst Five Records:")
print(df.head())

# Step 3: Display Dataset Information
print("\nFeature Names:")
print(feature_names)
print("\nTarget Names:")
print(target_names)
print("\nClass Distribution:")
print(df["Target"].value_counts())
print("\nDataset Description:")
print(df.describe())

# Step 4: Separate Features and Target
X = df[feature_names].values
y = df["Target"].values

# Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# Step 6: Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 7: Create SVM Model (RBF Kernel)
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    random_state=42
)

# Step 8: Train the Model
svm_model.fit(X_train_scaled, y_train)
print("\nSVM Model Training Completed Successfully.")

# Step 9: Display Support Vector Information
print("\n===================================")
print("SUPPORT VECTOR INFORMATION")
print("===================================")
print("Number of Support Vectors:", svm_model.n_support_)
print("Total Support Vectors:", len(svm_model.support_vectors_))

# Step 10: Make Predictions
y_pred = svm_model.predict(X_test_scaled)

# Step 11: Prediction Probabilities
y_probability = svm_model.predict_proba(X_test_scaled)

# Step 12: Decision Function Values
decision_values = svm_model.decision_function(X_test_scaled)

# Step 13: Actual vs Predicted
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Decision_Value": decision_values,
    "Probability_Benign": y_probability[:, 1]
})
print("\n===================================")
print("ACTUAL VS PREDICTED")
print("===================================")
print(results.head(20))

# Steps 14-17: Accuracy, Precision, Recall, F1
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

# Step 18: Display Performance
print("\n===================================")
print("SVM CLASSIFICATION PERFORMANCE")
print("===================================")
print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

# Step 19: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n===================================")
print("CONFUSION MATRIX")
print("===================================")
print(cm)

# Step 20: Classification Report
print("\n===================================")
print("CLASSIFICATION REPORT")
print("===================================")
print(classification_report(y_test, y_pred, target_names=target_names))

# Step 21: Plot Confusion Matrix
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("SVM Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(target_names))
plt.xticks(tick_marks, target_names)
plt.yticks(tick_marks, target_names)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()

# Step 22: Actual vs Predicted Plot
number_of_samples = min(30, len(y_test))
sample_index = np.arange(number_of_samples)

plt.figure(figsize=(10, 5))
plt.scatter(sample_index, y_test[:number_of_samples], marker="o", label="Actual Class")
plt.scatter(sample_index, y_pred[:number_of_samples], marker="x", label="Predicted Class")
plt.xlabel("Test Sample Index")
plt.ylabel("Class")
plt.title("SVM: Actual vs Predicted Classes")
plt.yticks([0, 1], target_names)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 23: Predict a New Sample
new_sample = X_test[0].reshape(1, -1)
new_sample_scaled = scaler.transform(new_sample)
new_prediction = svm_model.predict(new_sample_scaled)
new_probability = svm_model.predict_proba(new_sample_scaled)

print("\n===================================")
print("NEW SAMPLE PREDICTION")
print("===================================")
print("Predicted Class:", target_names[new_prediction[0]])
print("\nPrediction Probabilities:")
for i in range(len(target_names)):
    print(f"{target_names[i]}: {new_probability[0][i]:.4f}")